# 05 - End to end
Runs the full pipeline the way the app and API do: claim in, verdict and fallacies out.

In [ ]:
import os, sys, subprocess
if "google.colab" in sys.modules:  # Colab: clone the repo and install dependencies
    if not os.path.isdir("fact-and-logic"):
        subprocess.run(["git", "clone", "https://github.com/Anishkumar25/fact-and-logic.git"], check=True)
    os.chdir("fact-and-logic")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'httpx'], check=True)

In [ ]:
from src.pipeline import get_pipeline

p = get_pipeline()  # fallacy model is optional; the fact-checker works without it
claims = ['0-dimensional biomaterials lack inductive properties.',
          '1 in 5 million in UK have abnormal PrP positivity.',
          'Everyone I know got better after taking it, so it must work.']
for c in claims:
    r = p.run(c)
    print(c, '\n  ', r['verdict']['label'], round(r['verdict']['confidence'], 2), '|', r['verdict']['evidence'][:100], '\n  ', r['fallacies'][:2], '\n')

## Latency
This is what a user waits for on CPU.

In [ ]:
import time
t = time.perf_counter()
for c in claims: p.run(c)
print(f'{(time.perf_counter() - t) / len(claims):.2f} s per claim')

## API smoke test
Uses FastAPI's in-process test client, so no server needs to be running.

In [ ]:
from fastapi.testclient import TestClient
from src.api import app

c = TestClient(app)
print(c.get('/health').json())
c.post('/verify', json={'text': claims[0]}).json()